# NeuralPOD-DeepONet


In [ ]:
import os, sys, pathlib, json
import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt

sys.path.insert(0, str(pathlib.Path("..").resolve()))
from utils.datasets import load
from models.regime_basis import FourierRegimeBasis
from models.fourier_neural_pod import FourierNeuralPODTrainer, FourierNeuralPODConfig
from models.pod_deeponet import BranchNet
from models.neural_pod_deeponet import NeuralPODDeepONet, NeuralPODDeepONetConfig, NeuralPODDeepONetTrainer
from models.pod import PODTrainer, PODConfig
from models.pod_deeponet import PODDeepONet, PODDeepONetConfig, PODDeepONetTrainer, BranchNet

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

def rel_l2(true, pred):
    return np.linalg.norm(true - pred, axis=1) / np.linalg.norm(true, axis=1)

### Configuration

In [ ]:
# Run
RUN_NAME = "npod_deeponet_nu1.0_v1"
RUN_DIR = os.path.join("../../TEMPO_results", RUN_NAME)

# Data
TRAIN_NU = 1.0
ALL_NU = [0.001, 0.01, 0.1, 1.0]
N_TRAIN = 2500
N_TEST = 500

# Fourier Neural POD basis
MAX_MODES = 20
N_EPOCHS_MEAN = 800
N_EPOCHS_MODE = 1200
HIDDEN_DIM_BASIS = 256
NUM_FREQUENCIES = 96
SCALES = [0.5, 2.0, 6.0]
N_LAYERS_BASIS = 3

# Branch network
HIDDEN_DIM = 256
N_LAYERS = 5
SENSOR_STRIDE = 1

# Training
N_EPOCHS = 80000
BATCH_SIZE = 512

# Misc
SEED = 42
N_VIZ = 3

os.makedirs(RUN_DIR, exist_ok=True)
C0, C1, C2 = plt.cm.tab10(0), plt.cm.tab10(1), plt.cm.tab10(2)
print(f"Run dir: {os.path.abspath(RUN_DIR)}")

**Problem.** Given initial condition $u_0(x) = u(x, 0)$, approximate the solution operator
$\mathcal{G}: u_0 \mapsto u(\cdot, \cdot)$ for a time-dependent PDE on $x \in \Omega$, $t \in [0, T]$.

NeuralPOD-DeepONet replaces the linear POD basis with a nonlinear Fourier basis,
retaining the two-phase training structure of POD-DeepONet (Lu et al., 2022).


#### Phase 1: Fourier Neural POD


Let $\{u^n\}_{n=1}^N$ be training trajectories, each flattened to $\mathbf{s}^n \in \mathbb{R}^{N_t N_x}$.
The spatiotemporal coordinate of point $j$ is $\mathbf{c}_j = (x_j, t_j) \in \mathbb{R}^2$.

The field is approximated as a sum of $K$ separable modes:
$$\mathbf{s}^n \approx \boldsymbol{\mu}(\mathbf{c}) + \sum_{k=1}^K \lambda_k^n \, \boldsymbol{\phi}_k(\mathbf{c})$$

where:
- $\boldsymbol{\mu}: \mathbb{R}^2 \to \mathbb{R}$ is a neural mean field, trained first on the empirical mean
- $\boldsymbol{\phi}_k: \mathbb{R}^2 \to \mathbb{R}$ is the $k$-th spatial mode, parameterized via Random Fourier Features:
$$\boldsymbol{\phi}_k(\mathbf{c}) = \mathrm{MLP}\bigl([\sin(2\pi B\mathbf{c}),\, \cos(2\pi B\mathbf{c})]\bigr), \quad B \in \mathbb{R}^{F \times 2} \text{ fixed random}$$
- $\lambda_k^n \in \mathbb{R}$ is a scalar coefficient — one per trajectory per mode

Modes are extracted **greedily**: each $(\boldsymbol{\phi}_k, \{\lambda_k^n\})$ is trained to minimize the weighted residual
$$\mathcal{L}_k = \sum_{n=1}^N \gamma_n \sum_j w_j \bigl(\lambda_k^n \, \phi_k(\mathbf{c}_j) - r_j^n\bigr)^2$$
where $\mathbf{r}^n$ is the residual after subtracting the previous $k{-}1$ modes.
After training mode $k$, its parameters are frozen and the residual is deflated.


#### Phase 2: Branch Network


After Phase 1, the coefficient matrix $\Lambda \in \mathbb{R}^{N \times K}$ with $\Lambda_{nk} = \lambda_k^n$
encodes how each training trajectory projects onto the learned modes.

A branch network $\mathcal{B}_\theta: \mathbb{R}^m \to \mathbb{R}^K$ is trained to predict
these coefficients from the initial condition sampled at $m$ sensor points:
$$\hat{\boldsymbol{\lambda}}^n = \mathcal{B}_\theta\bigl(u_0^n(x_1), \ldots, u_0^n(x_m)\bigr)$$

Training minimizes MSE against Phase 1 coefficients:
$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{n=1}^N \bigl\|\mathcal{B}_\theta(u_0^n) - \boldsymbol{\lambda}^n\bigr\|_2^2$$

The mode networks $\boldsymbol{\phi}_k$ and mean $\boldsymbol{\mu}$ are not updated in this phase.

---

#### Prediction

For a new initial condition $u_0^*$, the full spatiotemporal field is:
$$\hat{u}(\mathbf{c}) = \boldsymbol{\mu}(\mathbf{c}) + \sum_{k=1}^K \bigl[\mathcal{B}_\theta(u_0^*)\bigr]_k \, \boldsymbol{\phi}_k(\mathbf{c})$$

Unlike POD-DeepONet, the modes are not stored as fixed vectors — they are evaluated
by forward passes through $\boldsymbol{\phi}_k$ at query coordinates $\mathbf{c}$.
The result is reshaped from $\mathbb{R}^{N_t N_x}$ to $(N_t, N_x)$.


### Data loading

In [ ]:
_local  = pathlib.Path("..") / "data" / f"Burgers_Nu{TRAIN_NU}.hdf5"
_server = pathlib.Path(os.path.expanduser("~/data/1D/Burgers/Train")) / f"1D_Burgers_Sols_Nu{TRAIN_NU}.hdf5"
path = str(_local) if _local.exists() else str(_server)
print(f"data: {path}")

with h5py.File(path, 'r') as f:
    raw = f['tensor'][:N_TRAIN + N_TEST]
    if raw.ndim == 4:
        raw = raw[..., 0]
    x_np = f['x-coordinate'][:]
    t_np = f['t-coordinate'][:]

N_total, Nt, Nx = raw.shape
t_np = t_np[:Nt]
print(f"loaded: N={N_total}, Nt={Nt}, Nx={Nx}")

In [ ]:
tensor_train = raw[:N_TRAIN]
tensor_test  = raw[N_TRAIN:]
del raw

In [ ]:
s_traj = torch.tensor(tensor_train.reshape(N_TRAIN, -1), dtype=torch.float32)
u0_train = torch.tensor(tensor_train[:, 0, :], dtype=torch.float32).to(DEVICE)
u0_test  = torch.tensor(tensor_test[:, 0, :],  dtype=torch.float32).to(DEVICE)

x_grid = torch.tensor(x_np, dtype=torch.float32)
t_grid = torch.tensor(t_np, dtype=torch.float32)
tt, xx = torch.meshgrid(t_grid, x_grid, indexing='ij')
x_flat = torch.stack([xx.flatten(), tt.flatten()], dim=1).to(DEVICE)

m = len(range(0, Nx, SENSOR_STRIDE))
print(f"s_traj: {tuple(s_traj.shape)}, x_flat: {tuple(x_flat.shape)}, m={m}")

### Phase 1: Fourier Neural POD training

In [ ]:
w = torch.ones(Nt * Nx, dtype=torch.float32).to(DEVICE) / (Nt * Nx)
basis = FourierRegimeBasis(
    d_x=2, M=N_TRAIN, quad_weights=w,
    hidden_dim=HIDDEN_DIM_BASIS,
    num_frequencies=NUM_FREQUENCIES,
    scales=SCALES,
    n_layers=N_LAYERS_BASIS,
).to(DEVICE)

cfg_pod = FourierNeuralPODConfig(
    max_modes=MAX_MODES,
    n_epochs_mean=N_EPOCHS_MEAN,
    n_epochs_mode=N_EPOCHS_MODE,
)
trainer_pod = FourierNeuralPODTrainer(basis, cfg_pod)
history_pod = trainer_pod.train(s_traj, x_flat, t=None)

### Phase 2: Branch network training

In [ ]:
K = basis.num_modes
branch = BranchNet(m=m, P=K, hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS).to(DEVICE)
model = NeuralPODDeepONet(basis, branch).to(DEVICE)

cfg = NeuralPODDeepONetConfig(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, sensor_stride=SENSOR_STRIDE)
trainer = NeuralPODDeepONetTrainer(model, cfg)
history_branch = trainer.train(u0_train)

### Results

In [ ]:
# Phase 1: classical POD (for comparison)
pod_trainer = PODTrainer(PODConfig(max_modes=32, tol=0.9999))
pod_trainer.train(s_traj, x_flat, t=None)
print(f"POD modes: {pod_trainer.num_modes}")

coeffs = pod_trainer.basis.coeffs.numpy()
modes = pod_trainer.basis.modes.numpy()
mean = pod_trainer.basis.mean.numpy()
s_pred_pod = mean + coeffs @ modes.T
err_pod_p1 = rel_l2(s_traj.numpy(), s_pred_pod)
print(f"POD Phase 1 reconstruction | mean={err_pod_p1.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(history_pod.residual_norms, 'o-', color=C0, markersize=4, lw=1.5)
ax.set_xlabel('Mode index')
ax.set_ylabel('Weighted residual norm')
ax.set_title('Phase 1: residual per mode', fontweight='bold')
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
logs = [v for sublist in history_pod.mode_losses for v in sublist]
ax.semilogy(logs, color=C0, lw=0.8)
lengths = [len(ml) for ml in history_pod.mode_losses]
boundaries = [sum(lengths[:i]) for i in range(1, len(lengths) + 1)]
for b in boundaries[:-1]:
    ax.axvline(b, color='gray', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel('Epoch (all modes concatenated)')
ax.set_ylabel('Mode training loss')
ax.set_title('Phase 1: mode training losses', fontweight='bold')
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "npod_phase1.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(history_branch, color=C0, lw=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Coefficient MSE')
ax.set_title('Phase 2: branch network training loss', fontweight='bold')
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "training_dynamics.png"), dpi=150, bbox_inches="tight")
plt.show()

### Evaluation

In [ ]:
true_train = s_traj.cpu().numpy()
pred_train = trainer.predict(u0_train, x_flat).cpu().numpy()

true_test = tensor_test.reshape(N_TEST, -1)
pred_test = trainer.predict(u0_test, x_flat).cpu().numpy()

err_train = rel_l2(true_train, pred_train)
err_test  = rel_l2(true_test,  pred_test)

print(f"Train | mean={err_train.mean():.4f}  median={np.median(err_train):.4f}  std={err_train.std():.4f}")
print(f"Test  | mean={err_test.mean():.4f}  median={np.median(err_test):.4f}  std={err_test.std():.4f}  p95={np.percentile(err_test, 95):.4f}")

### Metrics

In [ ]:
metrics = {
    "run_name":     RUN_NAME,
    "n_modes":      int(K),
    "n_train":      N_TRAIN,
    "n_test":       N_TEST,
    "train_mean":   float(err_train.mean()),
    "train_median": float(np.median(err_train)),
    "train_std":    float(err_train.std()),
    "test_mean":    float(err_test.mean()),
    "test_median":  float(np.median(err_test)),
    "test_std":     float(err_test.std()),
    "test_p95":     float(np.percentile(err_test, 95)),
}
print(metrics)

### Cross-$\nu$ generalization

In [ ]:
DATA_DIR = pathlib.Path(os.path.expanduser("~/data/1D/Burgers/Train"))

cross_nu_metrics = {}
for nu in ALL_NU:
    fpath = DATA_DIR / f"1D_Burgers_Sols_Nu{nu}.hdf5"
    if not fpath.exists():
        print(f"  nu={nu:.3f}: file not found, skipping")
        continue

    with h5py.File(fpath, "r") as f:
        raw_nu = f["tensor"][-N_TEST:]
        if raw_nu.ndim == 4:
            raw_nu = raw_nu[..., 0]

    u0_nu = torch.tensor(raw_nu[:, 0, :], dtype=torch.float32).to(DEVICE)
    s_nu  = raw_nu.reshape(N_TEST, -1)

    pred_nu = trainer.predict(u0_nu, x_flat).cpu().numpy()
    err_nu  = rel_l2(s_nu, pred_nu)

    cross_nu_metrics[nu] = {
        "mean":   float(err_nu.mean()),
        "median": float(np.median(err_nu)),
        "std":    float(err_nu.std()),
        "p95":    float(np.percentile(err_nu, 95)),
    }
    tag = " (trained)" if nu == TRAIN_NU else ""
    print(f"  nu={nu:.3f}{tag}: mean={err_nu.mean():.4f}  median={np.median(err_nu):.4f}  std={err_nu.std():.4f}")

metrics["cross_nu"] = cross_nu_metrics

In [ ]:
nu_labels = [f"$\\nu={nu:.3f}$" for nu in cross_nu_metrics]
means   = [v["mean"]   for v in cross_nu_metrics.values()]
medians = [v["median"] for v in cross_nu_metrics.values()]

x_pos = np.arange(len(nu_labels))
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(x_pos - 0.2,  means,   0.35, label='Mean',   color=C0, alpha=0.85, linewidth=0)
ax.bar(x_pos + 0.15, medians, 0.35, label='Median', color=C1, alpha=0.85, linewidth=0)

trained_idx = list(cross_nu_metrics.keys()).index(TRAIN_NU)
ax.axvline(trained_idx, color='gray', ls='--', lw=1.2, alpha=0.7, label='Trained on')

ax.set_xticks(x_pos)
ax.set_xticklabels(nu_labels)
ax.set_ylabel('Relative $L^2$ error')
ax.set_title('NeuralPOD-DeepONet — cross-$\\nu$ generalization', fontweight='bold')
ax.legend(framealpha=0.7)
ax.grid(True, ls='--', alpha=0.25, axis='y')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "cross_nu.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
idxs = rng.choice(N_TEST, size=N_VIZ, replace=False)

fig, axes = plt.subplots(N_VIZ, 3, figsize=(14, 3 * N_VIZ))
for row, idx in enumerate(idxs):
    pred = trainer.predict(u0_test[idx:idx+1], x_flat).reshape(Nt, Nx).cpu().numpy()
    true = tensor_test[idx]
    err  = np.abs(true - pred)
    vmax = np.abs(true).max()
    rl2  = np.linalg.norm(true - pred) / np.linalg.norm(true)

    for col, (arr, title, cmap, vmin, vm) in enumerate([
        (true, 'Ground Truth',         'RdBu_r', -vmax, vmax),
        (pred, 'NeuralPOD-DeepONet',   'RdBu_r', -vmax, vmax),
        (err,  'Absolute Error',        'Oranges',  0,   err.max()),
    ]):
        ax = axes[row, col]
        im = ax.imshow(arr, aspect='auto', origin='lower', cmap=cmap,
                       extent=[x_np.min(), x_np.max(), t_np.min(), t_np.max()],
                       vmin=vmin, vmax=vm)
        if row == 0:
            ax.set_title(title, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f't   (rel L2={rl2:.3f})')
        if row == N_VIZ - 1:
            ax.set_xlabel('x')
        ax.spines[['top', 'right']].set_visible(False)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('NeuralPOD-DeepONet: reconstruction examples', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "npod_reconstruction.png"), dpi=150, bbox_inches="tight")
plt.show()

### Reconstruction examples

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(err_test, bins=30, color=C0, alpha=0.8, linewidth=0)
ax.axvline(err_test.mean(),     color=C1, ls='--', lw=1.5, label=f'Mean {err_test.mean():.4f}')
ax.axvline(np.median(err_test), color=C2, ls='--', lw=1.5, label=f'Median {np.median(err_test):.4f}')
ax.set_xlabel('Relative $L^2$ error')
ax.set_ylabel('Count')
ax.set_title('Test error distribution', fontweight='bold')
ax.legend(framealpha=0.7)
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.plot(np.sort(err_test), color=C0, lw=1.5)
ax.axhline(err_test.mean(), color=C1, ls='--', lw=1.2, alpha=0.8)
ax.set_xlabel('Trajectory rank')
ax.set_ylabel('Relative $L^2$ error')
ax.set_title('Sorted test errors', fontweight='bold')
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('NeuralPOD-DeepONet — test set errors', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "npod_err_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

### Checkpoint

In [ ]:
torch.save({
    "model": model.state_dict(),
    "basis": basis.state_dict(),
    "cfg": cfg,
    "cfg_pod": cfg_pod,
    "metrics": metrics,
    "run_name": RUN_NAME,
}, os.path.join(RUN_DIR, "model.pt"))

with open(os.path.join(RUN_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved to {os.path.abspath(RUN_DIR)}")

### Error distribution